# Workshop 1 — Preprocessing, Model Training & Serialization
**Module:** From Data to Deployment: Python, APIs & ML  
**Dataset:** Titanic — Kaggle (891 passengers, binary classification)  
**Deliverable:** `../api/model.pkl` + this notebook  

Pipeline: `train.csv -> explore -> clean -> encode -> train DecisionTree -> tune -> save model.pkl`

---
**How to use:** Run cells top to bottom. Each section builds on the previous one.

## Section 0 — Install & Import

In [1]:
# Install required libraries (skip if already installed in your environment)
!pip install pandas scikit-learn --quiet

In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Section 1 — Load & Explore the Data

In [3]:
# Load the raw Titanic training dataset
df = pd.read_csv('../data/train.csv')

print(f"Dataset shape: {df.shape}")   # rows x columns
print(f"\nColumn names: {df.columns.tolist()}")
df.head()

Dataset shape: (891, 12)

Column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Basic statistics -- understand the range and distribution of each column
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
# Check for missing values -- critical before any modelling step
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_summary[missing_summary['Missing Count'] > 0])

          Missing Count  Missing %
Age                 177       19.9
Cabin               687       77.1
Embarked              2        0.2


In [6]:
# Check class balance -- are Survived/Not Survived roughly equal?
# Imbalanced classes can bias a model toward the majority class.
print("Survival counts:")
print(df['Survived'].value_counts())
print(f"\nSurvival rate: {df['Survived'].mean():.1%}")   # ~38% survived

Survival counts:
Survived
0    549
1    342
Name: count, dtype: int64

Survival rate: 38.4%


In [7]:
# Survival rate broken down by key categorical features
# This tells us which features are likely to be predictive

print("Survival rate by Sex:")
print(df.groupby('Sex')['Survived'].mean().round(3))

print("\nSurvival rate by Pclass:")
print(df.groupby('Pclass')['Survived'].mean().round(3))

print("\nSurvival rate by Embarked:")
print(df.groupby('Embarked')['Survived'].mean().round(3))

Survival rate by Sex:
Sex
female    0.742
male      0.189
Name: Survived, dtype: float64

Survival rate by Pclass:
Pclass
1    0.630
2    0.473
3    0.242
Name: Survived, dtype: float64

Survival rate by Embarked:
Embarked
C    0.554
Q    0.390
S    0.337
Name: Survived, dtype: float64


## Section 2 — Clean & Preprocess

Steps:
1. Drop columns that are too noisy or have too many missing values
2. Impute remaining missing values
3. Encode categorical columns as numbers (Decision Trees need numeric input)

In [8]:
# Work on a copy so the raw df stays intact for reference
df_clean = df.copy()

# -- Drop irrelevant columns ---------------------------------------------------
# PassengerId : just a row number, no predictive value
# Name        : free text -- too high cardinality for a simple tree
# Ticket      : alphanumeric codes with no consistent meaning
# Cabin       : 77% missing -- too sparse to be useful
df_clean = df_clean.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# -- Impute missing values -----------------------------------------------------
# Age is missing for ~20% of passengers.
# We use the median (not mean) because Age is right-skewed.
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)
print(f"Age median used for imputation: {age_median}")

# Embarked is missing for only 2 passengers -- fill with the most common port.
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)
print(f"Embarked mode used for imputation: {embarked_mode}")

# -- Encode categorical columns ------------------------------------------------
# Decision Trees need numeric input.
# Sex: male -> 0, female -> 1
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})

# Embarked: C -> 0, Q -> 1, S -> 2  (arbitrary but consistent)
df_clean['Embarked'] = df_clean['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

# Pclass is already numeric (1/2/3) -- no change needed

# -- Verify -------------------------------------------------------------------
print(f"\nNulls remaining: {df_clean.isnull().sum().sum()}")  # must be 0
df_clean.head()

Age median used for imputation: 28.0
Embarked mode used for imputation: S

Nulls remaining: 0


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,2
1,1,1,1,38.0,1,0,71.2833,0
2,1,3,1,26.0,0,0,7.9250,2
3,1,1,1,35.0,1,0,53.1000,2
4,0,3,0,35.0,0,0,8.0500,2


In [9]:
# Confirm dtypes -- all columns should now be numeric
print("Data types after encoding:")
print(df_clean.dtypes)
print(f"\nFinal shape: {df_clean.shape}")

Data types after encoding:
Survived      int64
Pclass        int64
Sex           int64
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked      int64
dtype: object

Final shape: (891, 8)


## Section 3 — Train / Test Split

In [10]:
# The 7 features our model will learn from (matches the FastAPI /predict endpoint)
FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
TARGET   = 'Survived'

X = df_clean[FEATURES]
y = df_clean[TARGET]

# 80% training data, 20% held out for evaluation
# random_state=42 makes the split reproducible (same split every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {X_train.shape[0]} rows")
print(f"Test set     : {X_test.shape[0]} rows")
print(f"\nClass balance in training set:")
print(y_train.value_counts(normalize=True).round(3))

Training set : 712 rows
Test set     : 179 rows

Class balance in training set:
Survived
0    0.624
1    0.376
Name: proportion, dtype: float64


## Section 4 — Model Training & Tuning

We train a **Decision Tree Classifier** and tune `max_depth` -- the single most important hyperparameter.

- **Too shallow** (depth 1-2): underfits, misses patterns  
- **Too deep** (None = unlimited): overfits the training data, fails on unseen data  
- **Sweet spot**: highest CV accuracy with a small overfit gap

We use **5-fold cross-validation** so every result is averaged over 5 different train/val splits -- more reliable than a single train/test split.

In [11]:
# -- Hyperparameter search: iterate over max_depth values ---------------------
# For each depth we record:
#   train_acc    -- how well the model fits the training data
#   cv_acc       -- 5-fold cross-validation accuracy (more honest estimate)
#   test_acc     -- accuracy on the held-out test set
#   overfit_gap  -- train_acc minus cv_acc; large gap = overfitting

depths = [2, 3, 4, 5, 6, 7, 8, None]   # None = unlimited depth
results = []

for depth in depths:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, m.predict(X_train))
    cv_acc    = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy').mean()
    test_acc  = accuracy_score(y_test, m.predict(X_test))

    results.append({
        'max_depth'  : str(depth),
        'train_acc'  : round(train_acc, 4),
        'cv_acc'     : round(cv_acc, 4),
        'test_acc'   : round(test_acc, 4),
        'overfit_gap': round(train_acc - cv_acc, 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

max_depth  train_acc  cv_acc  test_acc  overfit_gap
        2     0.8034  0.7865    0.7654       0.0169
        3     0.8343  0.8202    0.7989       0.0141
        4     0.8399  0.8033    0.7989       0.0365
        5     0.8511  0.8076    0.7989       0.0435
        6     0.8666  0.7991    0.8045       0.0675
        7     0.8834  0.8047    0.8045       0.0787
        8     0.8961  0.7991    0.7877       0.0970
     None     0.9789  0.7598    0.7821       0.2191


In [12]:
# Pick the best depth: highest CV accuracy (most reliable signal).
# If multiple depths tie, a shallower tree is preferred (simpler = less overfitting).
best_idx   = results_df['cv_acc'].idxmax()
best_row   = results_df.loc[best_idx]
best_depth = None if best_row['max_depth'] == 'None' else int(best_row['max_depth'])

print(f"Best max_depth  : {best_depth}")
print(f"CV accuracy     : {best_row['cv_acc']}")
print(f"Test accuracy   : {best_row['test_acc']}")
print(f"Overfit gap     : {best_row['overfit_gap']}  (train - CV; lower is better)")

Best max_depth  : 3
CV accuracy     : 0.8202
Test accuracy   : 0.7989
Overfit gap     : 0.0141  (train - CV; lower is better)


In [13]:
# -- Train the final model on the full training set ---------------------------
# Now that we have chosen the best depth, retrain on ALL training data
# (not just 4/5 of it as in cross-validation) for maximum data usage.
model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("=" * 42)
print(f"  Final model   max_depth = {best_depth}")
print("=" * 42)
print(f"\nTest accuracy : {accuracy_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion matrix:")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0  (died)    {cm[0][0]:>5}         {cm[0][1]:>5}")
print(f"  Actual 1  (surv)    {cm[1][0]:>5}         {cm[1][1]:>5}")

print(f"\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

  Final model   max_depth = 3

Test accuracy : 0.7989

Confusion matrix:
                 Predicted 0   Predicted 1
  Actual 0  (died)       92            13
  Actual 1  (surv)       23            51

Classification report:
                 precision    recall  f1-score   support

Did Not Survive       0.80      0.88      0.84       105
       Survived       0.80      0.69      0.74        74

       accuracy                           0.80       179
      macro avg       0.80      0.78      0.79       179
   weighted avg       0.80      0.80      0.80       179



In [14]:
# -- Feature importances ------------------------------------------------------
# Shows which features the tree relied on most when making splits.
# Higher importance = more useful for predicting survival.
importances = pd.Series(model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False).round(4)

print("Feature importances (higher = more useful to the model):")
print(importances.to_string())

Feature importances (higher = more useful to the model):
Sex         0.6057
Pclass      0.2095
Age         0.0754
Fare        0.0612
SibSp       0.0481
Parch       0.0000
Embarked    0.0000


## Section 5 — Save model.pkl

We serialize the trained model with `pickle` so the FastAPI service can load it at startup without retraining every time.

In [15]:
MODEL_PATH = '../api/model.pkl'

# Save the trained model to disk
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to {MODEL_PATH}")

# -- Reload check -------------------------------------------------------------
# Verify the file can be loaded back and produces the same predictions.
with open(MODEL_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

reload_acc = accuracy_score(y_test, loaded_model.predict(X_test))
print(f"Reload check -- accuracy: {reload_acc:.4f}  (should match above)")

# -- Manual prediction example ------------------------------------------------
# A 3rd-class male, age 22, 1 sibling, 0 parents, fare 7.25, embarked at S
# (roughly 'Jack' from the film -- expected: Did Not Survive)
sample = pd.DataFrame([{
    'Pclass': 3, 'Sex': 0, 'Age': 22,
    'SibSp': 1, 'Parch': 0, 'Fare': 7.25, 'Embarked': 2
}])

pred = loaded_model.predict(sample)[0]
print(f"\nSample prediction (Jack) -> {'Survived' if pred == 1 else 'Did Not Survive'}")

Model saved to ../api/model.pkl
Reload check -- accuracy: 0.7989  (should match above)

Sample prediction (Jack) -> Did Not Survive
